In [16]:
import pandas as pd
import numpy as np
import os

In [17]:
DATA_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"

os.makedirs(PROCESSED_PATH, exist_ok=True)

In [18]:
plant1_generation = pd.read_csv(
    DATA_PATH + "Plant_1_Generation_Data.csv"
)

plant1_weather = pd.read_csv(
    DATA_PATH + "Plant_1_Weather_Sensor_Data.csv"
)

plant2_generation = pd.read_csv(
    DATA_PATH + "Plant_2_Generation_Data.csv"
)

plant2_weather = pd.read_csv(
    DATA_PATH + "Plant_2_Weather_Sensor_Data.csv"
)

In [19]:
# Convert DATE_TIME using the correct format for each dataset

plant1_generation["DATE_TIME"] = pd.to_datetime(
    plant1_generation["DATE_TIME"],
    format="%d-%m-%Y %H:%M",
    errors="coerce"
)

plant2_generation["DATE_TIME"] = pd.to_datetime(
    plant2_generation["DATE_TIME"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

plant1_weather["DATE_TIME"] = pd.to_datetime(
    plant1_weather["DATE_TIME"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

plant2_weather["DATE_TIME"] = pd.to_datetime(
    plant2_weather["DATE_TIME"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

In [20]:
print(plant1_generation["DATE_TIME"].head())
print()
print(plant1_weather["DATE_TIME"].head())

0   2020-05-15
1   2020-05-15
2   2020-05-15
3   2020-05-15
4   2020-05-15
Name: DATE_TIME, dtype: datetime64[us]

0   2020-05-15 00:00:00
1   2020-05-15 00:15:00
2   2020-05-15 00:30:00
3   2020-05-15 00:45:00
4   2020-05-15 01:00:00
Name: DATE_TIME, dtype: datetime64[us]


In [21]:
# Check whether DATE_TIME was successfully converted to datetime
for name, df in {
    "Plant 1 Generation": plant1_generation,
    "Plant 1 Weather": plant1_weather,
    "Plant 2 Generation": plant2_generation,
    "Plant 2 Weather": plant2_weather
}.items():

    print(name)
    print("Data type:", df["DATE_TIME"].dtype)
    print("Invalid dates:", df["DATE_TIME"].isna().sum())
    print()

Plant 1 Generation
Data type: datetime64[us]
Invalid dates: 0

Plant 1 Weather
Data type: datetime64[us]
Invalid dates: 0

Plant 2 Generation
Data type: datetime64[us]
Invalid dates: 0

Plant 2 Weather
Data type: datetime64[us]
Invalid dates: 0



In [22]:
# Sort each dataset chronologically
plant1_generation = plant1_generation.sort_values(
    ["SOURCE_KEY", "DATE_TIME"]
).reset_index(drop=True)

plant1_weather = plant1_weather.sort_values(
    "DATE_TIME"
).reset_index(drop=True)

plant2_generation = plant2_generation.sort_values(
    ["SOURCE_KEY", "DATE_TIME"]
).reset_index(drop=True)

plant2_weather = plant2_weather.sort_values(
    "DATE_TIME"
).reset_index(drop=True)

Why SOURCE_KEY for generation?
Because we have 22 inverters per plant.
    We want:
Inverter A → chronological order
Inverter B → chronological order
Inverter C → chronological order
...

rather than mixing all inverters together.

In [23]:
# Check the first few timestamps after sorting
print("Plant 1 Generation:")
print(plant1_generation[["SOURCE_KEY", "DATE_TIME"]].head(10))

print("\nPlant 1 Weather:")
print(plant1_weather[["DATE_TIME"]].head(10))

Plant 1 Generation:
        SOURCE_KEY           DATE_TIME
0  1BY6WEcLGh8j5v7 2020-05-15 00:00:00
1  1BY6WEcLGh8j5v7 2020-05-15 00:15:00
2  1BY6WEcLGh8j5v7 2020-05-15 00:30:00
3  1BY6WEcLGh8j5v7 2020-05-15 00:45:00
4  1BY6WEcLGh8j5v7 2020-05-15 01:00:00
5  1BY6WEcLGh8j5v7 2020-05-15 01:15:00
6  1BY6WEcLGh8j5v7 2020-05-15 01:30:00
7  1BY6WEcLGh8j5v7 2020-05-15 01:45:00
8  1BY6WEcLGh8j5v7 2020-05-15 02:00:00
9  1BY6WEcLGh8j5v7 2020-05-15 02:15:00

Plant 1 Weather:
            DATE_TIME
0 2020-05-15 00:00:00
1 2020-05-15 00:15:00
2 2020-05-15 00:30:00
3 2020-05-15 00:45:00
4 2020-05-15 01:00:00
5 2020-05-15 01:15:00
6 2020-05-15 01:30:00
7 2020-05-15 01:45:00
8 2020-05-15 02:00:00
9 2020-05-15 02:15:00


In [24]:
# Check for physically unusual negative values
print("Plant 1 Generation")
print("Negative DC Power:",
      (plant1_generation["DC_POWER"] < 0).sum())
print("Negative AC Power:",
      (plant1_generation["AC_POWER"] < 0).sum())

print("\nPlant 2 Generation")
print("Negative DC Power:",
      (plant2_generation["DC_POWER"] < 0).sum())
print("Negative AC Power:",
      (plant2_generation["AC_POWER"] < 0).sum())

print("\nPlant 1 Weather")
print("Negative Irradiation:",
      (plant1_weather["IRRADIATION"] < 0).sum())

print("\nPlant 2 Weather")
print("Negative Irradiation:",
      (plant2_weather["IRRADIATION"] < 0).sum())

Plant 1 Generation
Negative DC Power: 0
Negative AC Power: 0

Plant 2 Generation
Negative DC Power: 0
Negative AC Power: 0

Plant 1 Weather
Negative Irradiation: 0

Plant 2 Weather
Negative Irradiation: 0


In [25]:
# Check the time gap between consecutive weather measurements
print("Plant 1 Weather:")
print(plant1_weather["DATE_TIME"].diff().value_counts().head(10))

print("\nPlant 2 Weather:")
print(plant2_weather["DATE_TIME"].diff().value_counts().head(10))

Plant 1 Weather:
DATE_TIME
0 days 00:15:00    3173
0 days 00:30:00       2
0 days 03:00:00       1
0 days 01:00:00       1
0 days 04:15:00       1
0 days 07:15:00       1
0 days 01:30:00       1
0 days 04:30:00       1
Name: count, dtype: int64

Plant 2 Weather:
DATE_TIME
0 days 00:15:00    3253
0 days 00:30:00       5
Name: count, dtype: int64


In [26]:
# Check time gaps within each inverter
print("Plant 1 Generation:")
print(
    plant1_generation
    .groupby("SOURCE_KEY")["DATE_TIME"]
    .diff()
    .value_counts()
    .head(10)
)

print("\nPlant 2 Generation:")
print(
    plant2_generation
    .groupby("SOURCE_KEY")["DATE_TIME"]
    .diff()
    .value_counts()
    .head(10)
)

Plant 1 Generation:
DATE_TIME
0 days 00:15:00    68424
0 days 01:00:00       52
0 days 00:30:00       46
0 days 01:30:00       37
0 days 01:45:00       35
0 days 00:45:00       33
0 days 04:15:00       22
0 days 09:00:00       22
0 days 08:00:00       22
0 days 03:00:00       20
Name: count, dtype: int64

Plant 2 Generation:
DATE_TIME
0 days 00:15:00    67556
0 days 00:30:00      106
0 days 16:15:00       10
8 days 18:30:00        4
Name: count, dtype: int64


In [27]:
# Check the actual time range after datetime conversion
for name, df in {
    "Plant 1 Generation": plant1_generation,
    "Plant 1 Weather": plant1_weather,
    "Plant 2 Generation": plant2_generation,
    "Plant 2 Weather": plant2_weather
}.items():

    print(name)
    print("Start:", df["DATE_TIME"].min())
    print("End  :", df["DATE_TIME"].max())
    print()

Plant 1 Generation
Start: 2020-05-15 00:00:00
End  : 2020-06-17 23:45:00

Plant 1 Weather
Start: 2020-05-15 00:00:00
End  : 2020-06-17 23:45:00

Plant 2 Generation
Start: 2020-05-15 00:00:00
End  : 2020-06-17 23:45:00

Plant 2 Weather
Start: 2020-05-15 00:00:00
End  : 2020-06-17 23:45:00



In [28]:
# Check duplicate timestamp + inverter combinations
print("Plant 1 Generation:")
print(
    plant1_generation.duplicated(
        subset=["DATE_TIME", "SOURCE_KEY"]
    ).sum()
)

print("\nPlant 2 Generation:")
print(
    plant2_generation.duplicated(
        subset=["DATE_TIME", "SOURCE_KEY"]
    ).sum()
)

Plant 1 Generation:
0

Plant 2 Generation:
0


In [29]:
# Check how many weather sensors/sources exist for each plant
print("Plant 1 Weather Sources:")
print(plant1_weather["SOURCE_KEY"].unique())

print("\nPlant 2 Weather Sources:")
print(plant2_weather["SOURCE_KEY"].unique())

Plant 1 Weather Sources:
<ArrowStringArray>
['HmiyD2TTLFNqkNe']
Length: 1, dtype: str

Plant 2 Weather Sources:
<ArrowStringArray>
['iq8k7ZNt4Mwm3w0']
Length: 1, dtype: str


In [30]:
# Create processed-data folder if it doesn't exist
os.makedirs(PROCESSED_PATH, exist_ok=True)

# Save cleaned datasets without changing the original raw files
plant1_generation.to_csv(
    PROCESSED_PATH + "plant1_generation_clean.csv",
    index=False
)

plant1_weather.to_csv(
    PROCESSED_PATH + "plant1_weather_clean.csv",
    index=False
)

plant2_generation.to_csv(
    PROCESSED_PATH + "plant2_generation_clean.csv",
    index=False
)

plant2_weather.to_csv(
    PROCESSED_PATH + "plant2_weather_clean.csv",
    index=False
)

print("All cleaned datasets saved successfully.")

All cleaned datasets saved successfully.
